# LLM Zoomcamp 2026 - Homework 2: Vector Search

Same 72 lesson pages as HW1 (commit `8c1834d`). We embed text with the lightweight
**ONNX** model (`Xenova/all-MiniLM-L6-v2`), then search by similarity: vector search
from scratch, `minsearch.VectorSearch`, keyword vs vector, and hybrid search (RRF).

**Setup**
```bash
uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
uv add --dev huggingface-hub jupyter
python download.py     # fetch the ONNX model
```

In [ ]:
import numpy as np
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch
from embedder import Embedder

emb = Embedder()  # loads models/Xenova/all-MiniLM-L6-v2

## Load the data (pinned to commit `8c1834d`, 72 pages)

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub", repo_name="llm-zoomcamp",
    commit_id="8c1834d", allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
len(documents)   # 72

## Q1. Embedding a query
Embed the query and read the first value of the 384-dim vector.

In [ ]:
query = "How does approximate nearest neighbor search work?"
v = emb.encode(query)
len(v), float(v[0])

## Q2. Cosine similarity
Vectors are normalized, so the dot product **is** the cosine similarity.

In [ ]:
target = next(d for d in documents
              if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md")
page_vec = emb.encode(target["content"])
float(page_vec.dot(v))

## Q3. Chunking and search by hand
Chunk the pages, embed every chunk, stack into `X`, and score against the query.

In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)
X = emb.encode_batch([c["content"] for c in chunks])
scores = X.dot(v)
chunks[int(np.argmax(scores))]["filename"]

## Q4. Vector search with minsearch

In [ ]:
vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

q4 = "What metric do we use to evaluate a search engine?"
vindex.search(emb.encode(q4), num_results=5)[0]["filename"]

## Q5. Text search vs vector search
Which file appears in the vector top-5 but not the text top-5?

In [ ]:
tindex = Index(text_fields=["content"], keyword_fields=["filename"])
tindex.fit(chunks)

q5 = "How do I store vectors in PostgreSQL?"
vec5 = vindex.search(emb.encode(q5), num_results=5)
txt5 = tindex.search(q5, num_results=5)

{r["filename"] for r in vec5} - {r["filename"] for r in txt5}

## Q6. Hybrid search with Reciprocal Rank Fusion (RRF)
Fuse the vector and text ranked lists by rank position (k = 60).

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores, docs = {}, {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

q6 = "How do I give the model access to tools?"
vector_results = vindex.search(emb.encode(q6), num_results=10)
text_results = tindex.search(q6, num_results=10)
rrf([vector_results, text_results])[0]["filename"]